# Federated SME Learning — FedAvg Across Nordic Sites

**HAIIP Phase 6 — Research Notebook**

Demonstrates privacy-preserving federated learning across:
- SME_FI: Finnish paper mill (vibration-heavy, humid)
- SME_SE: Swedish automotive stamping (high-cycle, temp extremes)
- SME_NO: Norwegian offshore pumps (corrosion-prone, saltwater)

**References**:
- McMahan et al. (2017) Communication-Efficient Learning of Deep Networks from Decentralized Data
- Li et al. (2020) Federated Learning: Challenges, Methods, and Future Directions

> **Experimental — not production-ready**. Simulated federation (no actual network).

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from haiip.core.federated import FederatedLearner
print('FederatedLearner ready')

## 1. Run FedAvg — 10 rounds, 3 epochs per round

In [ ]:
learner = FederatedLearner(random_state=42)
result  = learner.run(n_rounds=10, local_epochs=3)

print(f'Experiment ID:     {result.experiment_id}')
print(f'Total rounds:      {result.total_rounds}')
print(f'Federated F1:      {result.final_global_f1:.4f}')
print(f'Centralized F1:    {result.centralized_f1:.4f}')
print(f'Federated gap:     {result.federated_gap:+.4f}')
print(f'Privacy preserved: {result.privacy_preserved}')
if result.converged_at:
    print(f'Converged at round: {result.converged_at}')

## 2. Learning Curve — Global F1 per Round

In [ ]:
rounds      = [r.round_number for r in result.rounds]
global_f1s  = [r.global_f1   for r in result.rounds]
global_aucs = [r.global_auc  for r in result.rounds]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(rounds, global_f1s,  'o-', color='#1976d2', linewidth=2, label='Federated F1')
axes[0].axhline(result.centralized_f1, color='#d32f2f', linestyle='--', label=f'Centralized F1 ({result.centralized_f1:.3f})')
axes[0].set_xlabel('Round', fontsize=12)
axes[0].set_ylabel('F1 Score', fontsize=12)
axes[0].set_title('FedAvg Global F1 vs Centralized Baseline', fontsize=13)
axes[0].legend()
axes[0].set_ylim([0, 1])
axes[0].grid(alpha=0.3)

axes[1].plot(rounds, global_aucs, 's-', color='#388e3c', linewidth=2, label='Global AUC-ROC')
axes[1].axhline(result.centralized_auc, color='#f57c00', linestyle='--', label=f'Centralized AUC ({result.centralized_auc:.3f})')
axes[1].set_xlabel('Round', fontsize=12)
axes[1].set_ylabel('AUC-ROC', fontsize=12)
axes[1].set_title('FedAvg Global AUC-ROC', fontsize=13)
axes[1].legend()
axes[1].set_ylim([0, 1])
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/figures/federated_learning_curve.png', dpi=150)
plt.show()

## 3. Per-Node F1 Across Rounds

In [ ]:
node_colors = {'SME_FI': '#1565c0', 'SME_SE': '#2e7d32', 'SME_NO': '#bf360c'}
node_labels = {'SME_FI': 'FI (Paper Mill)', 'SME_SE': 'SE (Stamping)', 'SME_NO': 'NO (Offshore)'}

fig, ax = plt.subplots(figsize=(12, 6))
for node_id, color in node_colors.items():
    f1s = [r.node_f1s.get(node_id, 0) for r in result.rounds]
    ax.plot(rounds, f1s, 'o-', color=color, label=node_labels[node_id], linewidth=1.5, alpha=0.8)

ax.plot(rounds, global_f1s, 'k--', linewidth=2.5, label='Global (FedAvg)', zorder=5)
ax.set_xlabel('Round', fontsize=12)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('Per-Node Local F1 and Global FedAvg F1', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Node Profiles — Non-IID Distribution Comparison

In [ ]:
print('Node profiles:')
for node_id, profile in result.node_profiles.items():
    print(f"  {node_id}: n={profile['n_samples']:,}, failure_rate={profile['failure_rate']:.0%}, industry: {profile['industry']}")

print(f'\nFederated gap: {result.federated_gap:+.4f}')
print('(Positive gap = centralized is better — expected due to non-IID data)')
print('(Gap < 0.15 meets our research threshold for acceptable federation quality)')